# EmoExpress: LLM-Based Topic Classifier

## Objective

This notebook implements an LLM-based topic classifier for EmoExpress.

The classifier assigns each user story to one predefined primary topic
and, when appropriate, one secondary topic.

The topic results will later be combined with:

- The original user story
- Predicted emotions
- RAG document retrieval
- Empathetic response generation
- Image and caption generation

Import libraries

In [1]:
from pathlib import Path
import json
import os
import time

import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

Configure paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# print("Project root:", PROJECT_ROOT)
# print("Metrics directory:", METRICS_DIR)

Load the OpenAI API key

In [3]:
load_dotenv(
    PROJECT_ROOT / ".env"
)

api_key = os.getenv(
       "OPENAI_API_KEY"
)

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found in the .env file."
    )

client = OpenAI(
    api_key=api_key
)

print("OpenAI client initialized.")

OpenAI client initialized.


**Define the topic taxonomy**

Create the controlled topic list

In [4]:
TOPIC_DEFINITIONS = {
    "career_and_work": (
        "Jobs, interviews, unemployment, coworkers, "
        "workplace conflict, career decisions, or burnout."
    ),

    "education": (
        "School, college, exams, grades, assignments, "
        "studying, teachers, or academic pressure."
    ),

    "relationships": (
        "Romantic relationships, dating, breakups, "
        "trust, intimacy, or partner communication."
    ),

    "family": (
        "Parents, children, siblings, marriage, caregiving, "
        "or conflict involving family members."
    ),

    "stress_and_anxiety": (
        "General worry, nervousness, overwhelm, tension, "
        "uncertainty, or anxiety not better explained by "
        "another primary topic."
    ),

    "grief_and_loss": (
        "Death, bereavement, separation, major loss, "
        "or mourning."
    ),

    "loneliness": (
        "Social isolation, lack of connection, feeling alone, "
        "or feeling unsupported."
    ),

    "self_confidence": (
        "Self-esteem, insecurity, self-doubt, shame, "
        "negative self-talk, or feelings of inadequacy."
    ),

    "financial_concerns": (
        "Bills, debt, budgeting, income, housing costs, "
        "or financial insecurity."
    ),

    "health_and_wellness": (
        "Sleep, exercise, illness, physical health, routines, "
        "or general mental and emotional wellness."
    ),

    "personal_growth": (
        "Goals, resilience, habits, motivation, identity, "
        "self-improvement, or major life changes."
    ),

    "daily_life": (
        "Ordinary events, minor frustrations, hobbies, "
        "or general experiences that do not fit another topic."
    ),

    "other": (
        "The story is unclear or does not match any supported topic."
    ),
}

TOPIC_LABELS = list(
    TOPIC_DEFINITIONS.keys()
)

print("Number of topics:", len(TOPIC_LABELS))

for topic, definition in TOPIC_DEFINITIONS.items():
    print(f"{topic}: {definition}")

Number of topics: 13
career_and_work: Jobs, interviews, unemployment, coworkers, workplace conflict, career decisions, or burnout.
education: School, college, exams, grades, assignments, studying, teachers, or academic pressure.
relationships: Romantic relationships, dating, breakups, trust, intimacy, or partner communication.
family: Parents, children, siblings, marriage, caregiving, or conflict involving family members.
stress_and_anxiety: General worry, nervousness, overwhelm, tension, uncertainty, or anxiety not better explained by another primary topic.
grief_and_loss: Death, bereavement, separation, major loss, or mourning.
loneliness: Social isolation, lack of connection, feeling alone, or feeling unsupported.
self_confidence: Self-esteem, insecurity, self-doubt, shame, negative self-talk, or feelings of inadequacy.
financial_concerns: Bills, debt, budgeting, income, housing costs, or financial insecurity.
health_and_wellness: Sleep, exercise, illness, physical health, routines,

Define the model

In [5]:
TOPIC_CLASSIFIER_MODEL = "gpt-4o-mini"

print(
    "Topic-classifier model:",
    TOPIC_CLASSIFIER_MODEL,
)

Topic-classifier model: gpt-4o-mini


**Build the prompt**

Format topic definitions

In [6]:
def format_topic_definitions(
    topic_definitions
):
    return "\n".join(
        f"- {topic}: {definition}"
        for topic, definition
        in topic_definitions.items()
    )

In [7]:
formatted_topics = format_topic_definitions(
    TOPIC_DEFINITIONS
)

print(formatted_topics)

- career_and_work: Jobs, interviews, unemployment, coworkers, workplace conflict, career decisions, or burnout.
- education: School, college, exams, grades, assignments, studying, teachers, or academic pressure.
- relationships: Romantic relationships, dating, breakups, trust, intimacy, or partner communication.
- family: Parents, children, siblings, marriage, caregiving, or conflict involving family members.
- stress_and_anxiety: General worry, nervousness, overwhelm, tension, uncertainty, or anxiety not better explained by another primary topic.
- grief_and_loss: Death, bereavement, separation, major loss, or mourning.
- loneliness: Social isolation, lack of connection, feeling alone, or feeling unsupported.
- self_confidence: Self-esteem, insecurity, self-doubt, shame, negative self-talk, or feelings of inadequacy.
- financial_concerns: Bills, debt, budgeting, income, housing costs, or financial insecurity.
- health_and_wellness: Sleep, exercise, illness, physical health, routines, 

Create the system prompt

In [8]:
TOPIC_SYSTEM_PROMPT = f"""
You are the topic-classification component of EmoExpress.

Classify the user's story using only the allowed topic labels below.

ALLOWED TOPICS

{format_topic_definitions(TOPIC_DEFINITIONS)}

CLASSIFICATION RULES

1. Choose exactly one primary topic.
2. Choose no more than one secondary topic.
3. The secondary topic must be different from the primary topic.
4. Use null for secondary_topic when no second topic is strongly relevant.
5. Do not create new topic labels.
6. Classify the situation being discussed, not merely its emotion.
7. Use "other" only when none of the other labels reasonably applies.
8. Confidence must be a number from 0.0 to 1.0.
9. Keep the explanation to one short sentence.
10. Return valid JSON only.

REQUIRED JSON FORMAT

{{
    "primary_topic": "one allowed topic",
    "secondary_topic": "one allowed topic or null",
    "confidence": 0.0,
    "explanation": "brief reason"
}}
""".strip()

**Create the classifier**

In [9]:
def validate_topic_result(
    result
):
    required_fields = {
        "primary_topic",
        "secondary_topic",
        "confidence",
        "explanation",
    }

    missing_fields = (
        required_fields
        - set(result.keys())
    )

    if missing_fields:
        raise ValueError(
            f"Missing fields: {missing_fields}"
        )

    primary_topic = result[
        "primary_topic"
    ]

    secondary_topic = result[
        "secondary_topic"
    ]

    confidence = float(
        result["confidence"]
    )

    if primary_topic not in TOPIC_LABELS:
        raise ValueError(
            "Invalid primary topic: "
            f"{primary_topic}"
        )

    if (
        secondary_topic is not None
        and secondary_topic
        not in TOPIC_LABELS
    ):
        raise ValueError(
            "Invalid secondary topic: "
            f"{secondary_topic}"
        )

    if secondary_topic == primary_topic:
        secondary_topic = None

    confidence = max(
        0.0,
        min(1.0, confidence),
    )

    return {
        "primary_topic": primary_topic,
        "secondary_topic": secondary_topic,
        "confidence": confidence,
        "explanation": str(
            result["explanation"]
        ).strip(),
    }

Create the topic-classification function

In [10]:
def classify_topic(
    user_story,
    model=TOPIC_CLASSIFIER_MODEL,
):
    """
    Classify one user story into a controlled topic taxonomy.
    """

    if not isinstance(user_story, str):
        raise TypeError(
            "user_story must be a string."
        )

    user_story = user_story.strip()

    if not user_story:
        raise ValueError(
            "The user story cannot be empty."
        )

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system",
                "content": TOPIC_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    "Classify this user story:\n\n"
                    f"{user_story}"
                ),
            },
        ],
    )

    raw_content = (
        response
        .choices[0]
        .message
        .content
    )

    parsed_result = json.loads(
        raw_content
    )

    validated_result = validate_topic_result(
        parsed_result
    )

    validated_result["user_story"] = (
        user_story
    )

    return validated_result

In [11]:
# test one sample

sample_story = """
I have applied to many jobs, but I keep getting rejected.
I am starting to think that I am not good enough.
"""

sample_result = classify_topic(
    sample_story
)

sample_result

{'primary_topic': 'career_and_work',
 'secondary_topic': 'self_confidence',
 'confidence': 0.9,
 'explanation': 'The story primarily discusses job applications and rejections, while also expressing feelings of inadequacy.',
 'user_story': 'I have applied to many jobs, but I keep getting rejected.\nI am starting to think that I am not good enough.'}

*Test multiple user stories*

Create test stories

In [12]:
test_stories = [
    (
        "I have an important exam tomorrow, "
        "and I cannot stop worrying about failing."
    ),

    (
        "My partner and I keep arguing, and I feel "
        "like we no longer understand each other."
    ),

    (
        "I lost my grandmother last month and still "
        "feel overwhelmed whenever I think about her."
    ),

    (
        "My rent is due next week, but I do not have "
        "enough money to pay it."
    ),

    (
        "I spend most evenings alone and feel like "
        "I have nobody to talk to."
    ),

    (
        "I am trying to establish healthier habits "
        "and become more consistent with exercise."
    ),

    (
        "My manager criticized my work, and now I "
        "question whether I am capable of doing my job."
    ),

    (
        "I stayed up late watching a movie and feel "
        "tired today."
    ),
]

Classify the test stories

In [13]:
topic_results = []

for story in test_stories:
    try:
        start_time = time.perf_counter()

        result = classify_topic(
            story
        )

        result["inference_time_seconds"] = (
            time.perf_counter()
            - start_time
        )

        topic_results.append(
            result
        )

    except Exception as error:
        topic_results.append(
            {
                "user_story": story,
                "primary_topic": "error",
                "secondary_topic": None,
                "confidence": 0.0,
                "explanation": str(error),
                "inference_time_seconds": None,
            }
        )

In [14]:
# display
topic_results_df = pd.DataFrame(
    topic_results
)

topic_results_df[
    [
        "user_story",
        "primary_topic",
        "secondary_topic",
        "confidence",
        "explanation",
        "inference_time_seconds",
    ]
]

,user_story,primary_topic,secondary_topic,confidence,explanation,inference_time_seconds
0,"I have an important exam tomorrow, and I canno...",education,stress_and_anxiety,0.9,"The story revolves around an exam, which is an...",1.065534
1,"My partner and I keep arguing, and I feel like...",relationships,NaN,0.9,The story discusses conflicts and communicatio...,2.471582
2,I lost my grandmother last month and still fee...,grief_and_loss,NaN,0.9,The story discusses the emotional impact of lo...,0.981620
3,"My rent is due next week, but I do not have en...",financial_concerns,NaN,0.9,The story discusses a specific financial issue...,1.341914
4,I spend most evenings alone and feel like I ha...,loneliness,NaN,0.9,The user expresses feelings of social isolatio...,1.248770
5,I am trying to establish healthier habits and ...,personal_growth,NaN,0.9,The story focuses on self-improvement and esta...,1.025847
6,"My manager criticized my work, and now I quest...",self_confidence,career_and_work,0.9,The story reflects self-doubt stemming from wo...,0.879163
7,I stayed up late watching a movie and feel tir...,health_and_wellness,NaN,0.9,"The story relates to feeling tired, which is c...",1.237342


**Evaluate the classifier**

Create labeled evaluation examples

In [15]:
evaluation_examples = [
    {
        "story": (
            "I have a job interview tomorrow, "
            "and I am afraid I will fail."
        ),
        "expected_topic": "career_and_work",
    },

    {
        "story": (
            "My final exam is next week, but I cannot "
            "focus when I try to study."
        ),
        "expected_topic": "education",
    },

    {
        "story": (
            "My boyfriend stopped responding to my messages, "
            "and I do not know what happened."
        ),
        "expected_topic": "relationships",
    },

    {
        "story": (
            "My parents keep comparing me with my sister."
        ),
        "expected_topic": "family",
    },

    {
        "story": (
            "Nothing specific is wrong, but I feel worried "
            "and tense all day."
        ),
        "expected_topic": "stress_and_anxiety",
    },

    {
        "story": (
            "My close friend passed away, and I miss her "
            "every day."
        ),
        "expected_topic": "grief_and_loss",
    },

    {
        "story": (
            "I moved to a new city and have nobody "
            "to spend time with."
        ),
        "expected_topic": "loneliness",
    },

    {
        "story": (
            "I always compare myself with others and feel "
            "like I am not talented enough."
        ),
        "expected_topic": "self_confidence",
    },

    {
        "story": (
            "My credit-card balance keeps increasing, "
            "and I cannot afford all my bills."
        ),
        "expected_topic": "financial_concerns",
    },

    {
        "story": (
            "I have not been sleeping well and feel "
            "exhausted every morning."
        ),
        "expected_topic": "health_and_wellness",
    },

    {
        "story": (
            "I want to become more disciplined and stop "
            "giving up on my goals."
        ),
        "expected_topic": "personal_growth",
    },

    {
        "story": (
            "The grocery store was crowded today, "
            "and the long line annoyed me."
        ),
        "expected_topic": "daily_life",
    },
]

Run evaluation

In [16]:
evaluation_results = []

for example in evaluation_examples:
    story = example["story"]

    start_time = time.perf_counter()

    prediction = classify_topic(
        story
    )

    inference_time = (
        time.perf_counter()
        - start_time
    )

    evaluation_results.append(
        {
            "story": story,
            "expected_topic": example[
                "expected_topic"
            ],
            "predicted_topic": prediction[
                "primary_topic"
            ],
            "secondary_topic": prediction[
                "secondary_topic"
            ],
            "confidence": prediction[
                "confidence"
            ],
            "correct": (
                prediction["primary_topic"]
                == example["expected_topic"]
            ),
            "inference_time_seconds": (
                inference_time
            ),
        }
    )

In [17]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

,story,expected_topic,predicted_topic,secondary_topic,confidence,correct,inference_time_seconds
0,"I have a job interview tomorrow, and I am afra...",career_and_work,career_and_work,stress_and_anxiety,0.9,True,1.240711
1,"My final exam is next week, but I cannot focus...",education,education,stress_and_anxiety,0.9,True,1.035167
2,My boyfriend stopped responding to my messages...,relationships,relationships,NaN,0.9,True,1.099130
3,My parents keep comparing me with my sister.,family,family,NaN,0.9,True,1.014124
4,"Nothing specific is wrong, but I feel worried ...",stress_and_anxiety,stress_and_anxiety,NaN,0.9,True,1.189491
5,"My close friend passed away, and I miss her ev...",grief_and_loss,grief_and_loss,NaN,1.0,True,1.126591
6,I moved to a new city and have nobody to spend...,loneliness,loneliness,NaN,0.9,True,0.921441
7,I always compare myself with others and feel l...,self_confidence,self_confidence,NaN,0.9,True,1.505067
8,"My credit-card balance keeps increasing, and I...",financial_concerns,financial_concerns,NaN,0.9,True,1.068000
9,I have not been sleeping well and feel exhaust...,health_and_wellness,health_and_wellness,NaN,0.9,True,0.925650


Calculate accuracy

In [18]:
topic_accuracy = accuracy_score(
    evaluation_df["expected_topic"],
    evaluation_df["predicted_topic"],
)

print(
    f"Topic classification accuracy: "
    f"{topic_accuracy:.4f}"
)

Topic classification accuracy: 1.0000


Generate the classification report

In [19]:
used_topics = sorted(
    set(
        evaluation_df[
            "expected_topic"
        ]
    )
    | set(
        evaluation_df[
            "predicted_topic"
        ]
    )
)

topic_report = classification_report(
    evaluation_df["expected_topic"],
    evaluation_df["predicted_topic"],
    labels=used_topics,
    zero_division=0,
    output_dict=True,
)

topic_report_df = pd.DataFrame(
    topic_report
).transpose()

topic_report_df

,precision,recall,f1-score,support
career_and_work,1.0,1.0,1.0,1.0
daily_life,1.0,1.0,1.0,1.0
education,1.0,1.0,1.0,1.0
family,1.0,1.0,1.0,1.0
financial_concerns,1.0,1.0,1.0,1.0
grief_and_loss,1.0,1.0,1.0,1.0
health_and_wellness,1.0,1.0,1.0,1.0
loneliness,1.0,1.0,1.0,1.0
personal_growth,1.0,1.0,1.0,1.0
relationships,1.0,1.0,1.0,1.0


Inspect incorrect predictions

In [20]:
incorrect_predictions_df = evaluation_df[
    evaluation_df["correct"] == False
]

incorrect_predictions_df[
    [
        "story",
        "expected_topic",
        "predicted_topic",
        "secondary_topic",
        "confidence",
    ]
]

,story,expected_topic,predicted_topic,secondary_topic,confidence


**Add consistency testing**

Test repeated predictions. Because LLM outputs can vary, test the same story several times:

In [21]:
consistency_story = """
I keep getting rejected after job interviews,
and I am losing confidence in myself.
"""

consistency_results = []

for run_number in range(5):
    result = classify_topic(
        consistency_story
    )

    consistency_results.append(
        {
            "run": run_number + 1,
            "primary_topic": result[
                "primary_topic"
            ],
            "secondary_topic": result[
                "secondary_topic"
            ],
            "confidence": result[
                "confidence"
            ],
        }
    )

consistency_df = pd.DataFrame(
    consistency_results
)

consistency_df

,run,primary_topic,secondary_topic,confidence
0,1,career_and_work,self_confidence,0.9
1,2,career_and_work,self_confidence,0.9
2,3,career_and_work,self_confidence,0.9
3,4,career_and_work,self_confidence,0.9
4,5,career_and_work,self_confidence,0.9


**Create a safe batch function**

Add error handling

In [22]:
def classify_topic_safely(
    user_story
):
    try:
        return classify_topic(
            user_story
        )

    except json.JSONDecodeError:
        return {
            "user_story": user_story,
            "primary_topic": "other",
            "secondary_topic": None,
            "confidence": 0.0,
            "explanation": (
                "The model returned invalid JSON."
            ),
            "status": "invalid_json",
        }

    except Exception as error:
        return {
            "user_story": user_story,
            "primary_topic": "other",
            "secondary_topic": None,
            "confidence": 0.0,
            "explanation": str(error),
            "status": "error",
        }

In [23]:
classify_topic_safely(
    "I feel uncertain about what to do next."
)

{'primary_topic': 'stress_and_anxiety',
 'secondary_topic': None,
 'confidence': 0.8,
 'explanation': 'The user expresses feelings of uncertainty, which aligns with stress and anxiety.',
 'user_story': 'I feel uncertain about what to do next.'}

# Prepare the topic output for RAG

**Build a retrieval-query helper**

In [24]:
def build_topic_retrieval_query(
    user_story,
    emotion_predictions,
    topic_result,
):
    emotion_labels = [
        prediction["emotion"]
        for prediction in emotion_predictions
    ]

    secondary_topic = (
        topic_result["secondary_topic"]
        or "none"
    )

    query = f"""
User situation:
{user_story}

Detected emotions:
{", ".join(emotion_labels)}

Primary topic:
{topic_result["primary_topic"]}

Secondary topic:
{secondary_topic}

Retrieve safe, practical, non-diagnostic guidance,
coping strategies, and constructive next steps that
are directly relevant to this situation.
""".strip()

    return query

In [25]:
# Example
example_emotions = [
    {
        "emotion": "disappointment",
        "score": 0.82,
    },
    {
        "emotion": "sadness",
        "score": 0.69,
    },
]

example_topic = classify_topic(
    """
    I have failed several interviews and now feel
    like I am not good enough.
    """
)

retrieval_query = build_topic_retrieval_query(
    user_story=(
        "I have failed several interviews and now "
        "feel like I am not good enough."
    ),
    emotion_predictions=example_emotions,
    topic_result=example_topic,
)

print(retrieval_query)

User situation:
I have failed several interviews and now feel like I am not good enough.

Detected emotions:
disappointment, sadness

Primary topic:
career_and_work

Secondary topic:
self_confidence

Retrieve safe, practical, non-diagnostic guidance,
coping strategies, and constructive next steps that
are directly relevant to this situation.


**Save outputs**

Save evaluation results

In [26]:
evaluation_path = (
    METRICS_DIR
    / "topic_classifier_evaluation.csv"
)

evaluation_df.to_csv(
    evaluation_path,
    index=False,
)

# print(
#     "Saved evaluation:",
#     evaluation_path,
# )

Save the classification report:

In [27]:
topic_report_path = (
    METRICS_DIR
    / "topic_classifier_report.csv"
)

topic_report_df.to_csv(
    topic_report_path,
    index=True,
)

# print(
#     "Saved report:",
#     topic_report_path,
# )

Save the topic taxonomy

In [28]:
topic_taxonomy_path = (
    METRICS_DIR
    / "topic_taxonomy.json"
)

with open(
    topic_taxonomy_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        TOPIC_DEFINITIONS,
        file,
        indent=4,
    )

# print(
#     "Saved taxonomy:",
#     topic_taxonomy_path,
# )

# Conclusion

An LLM-based topic classifier was implemented using a controlled taxonomy
of 13 topic labels.

The classifier returns:

- One primary topic
- One optional secondary topic
- A confidence score
- A brief explanation

The primary topic represents the main situation described by the user,
while the secondary topic captures an important related concern.

A manually labeled evaluation set was used to test classification
accuracy and inspect incorrect predictions. Because the evaluation set
is relatively small, the results represent functional validation rather
than a definitive benchmark.

The topic output will next be combined with the original user story and
predicted emotions to construct a topic-aware RAG retrieval query.